## Imports

In [2]:
import torch
from transformers import BertTokenizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\35159\miniforge3\envs\autoregulatorycuda\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\35159\miniforge3\envs\autoregulatorycuda\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\35159\miniforge3\envs\autoregulatorycuda\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\35159\miniforge3\envs\autoregulatorycuda\lib\site-packages\traitlets\config\applicat

## Step 1: Read Data

In [3]:
df = pd.read_csv("data/processed_data/protein_autoregulatory_terms.csv")

df = df[df["Terms"].notna()]
df = df[df["Terms"].str.strip() != ""]

print(f"Number of Labeled Rows: {len(df)}")

df.head()

剩余可用样本数: 1823


,AC,OS,PMID,Title,Abstract,Terms
1085,P63104,Homo sapiens (Human),29357390.0,Herpesvirus deconjugases inhibit the IFN respo...,The N-terminal domains of the herpesvirus larg...,autoubiquitination
5416,Q64264,Mus musculus (Mouse),18599790.0,Sporadic autonomic dysregulation and death ass...,Sudden infant death syndrome is the leading ca...,autoinhibition
5891,Q9Y6E2,Homo sapiens (Human),29470543.0,Translational autoregulation of BZW1 and BZW2 ...,The efficiency of start codon selection during...,autoregulation
5926,Q7L1Q6,Homo sapiens (Human),29470543.0,Translational autoregulation of BZW1 and BZW2 ...,The efficiency of start codon selection during...,autoregulation
9655,Q13131,Homo sapiens (Human),17088252.0,Conserved alpha-helix acts as autoinhibitory s...,AMP-activated protein kinase (AMPK) acts as an...,autoinhibitory


## Step 2: Label Structure

In [4]:
from collections import Counter

all_terms = [
    term.strip().lower()
    for terms in df["Terms"]
    for term in terms.split(",")
    if term.strip()
]

term_counts = Counter(all_terms)

import pandas as pd
term_df = pd.DataFrame(term_counts.items(), columns=["Term", "Count"]).sort_values(by="Count", ascending=False)
term_df.head(10)

,Term,Count
5,autophosphorylation,867
6,autocatalytic,179
2,autoregulation,160
0,autoubiquitination,152
1,autoinhibition,138
8,autoregulatory,85
4,autoinducer,76
7,autolysis,71
3,autoinhibitory,61
9,autoactivation,26


## Step 3: Input Text

In [5]:
def combine_fields(row):
    return f"Protein ID: {row['AC']} | Species: {row['OS']} | Title: {row['Title']} | Abstract: {row['Abstract']}"
df["input_text"] = df.apply(combine_fields, axis=1)
texts = df["input_text"].tolist()

from sklearn.preprocessing import MultiLabelBinarizer
labels = df["Terms"].str.split(",").apply(lambda x: [t.strip().lower() for t in x])
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(labels)

print(f"Number of Labels: {len(mlb.classes_)}")

标签数量: 15


## Step 4: Splitting Train & Test Sets

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42
)

print(f"Training Set: {len(X_train)}")
print(f"Test Set: {len(X_test)}")

训练集样本数: 1458
测试集样本数: 365


## Step 5: Tokenization

In [7]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    X_train,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

test_encodings = tokenizer(
    X_test,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

## Step 6: Dataset Class

In [8]:
import torch

class ProteinDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = ProteinDataset(train_encodings, y_train)
test_dataset = ProteinDataset(test_encodings, y_test)

## Step 7: Multi Label Classifier

In [9]:
from transformers import BertModel
import torch.nn as nn

class BERTMultiLabelClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [CLS] token representation
        logits = self.classifier(self.dropout(pooled_output))
        return logits
    
model = BERTMultiLabelClassifier("bert-base-uncased", num_labels=y_train.shape[1])

## Step 8: Training Loop

In [40]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import f1_score
from tqdm import tqdm
import torch

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

criterion = BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

device = torch.device("cpu")
model.to(device)

EPOCHS = 3
print_every = 20  

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    print(f"\n Epoch {epoch + 1}/{EPOCHS}")

    for step, batch in enumerate(tqdm(train_loader, desc="Training")):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % print_every == 0:
            print(f"  Step {step+1}/{len(train_loader)} - Batch Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f" Epoch {epoch+1} Complete - Average Loss: {avg_loss:.4f}")


🟦 Epoch 1/3


Training:   2%|▏         | 2/92 [00:42<31:17, 20.86s/it]

## Step 9: Evaluation

In [41]:
def evaluate(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    f1 = f1_score(y_true, y_pred, average="micro")
    print(f"Micro F1 Score: {f1:.4f}")

evaluate(model, test_loader)

Micro F1 Score: 0.6186


In [42]:
torch.save(model.state_dict(), "bert_multilabel.pt")

In [43]:
model.load_state_dict(torch.load("bert_multilabel.pt"))
model.eval()

C:\Users\35159\AppData\Local\Temp\ipykernel_9468\1711685958.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("bert_multilabel.pt"))


BERTMultiLabelClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 